# KRM-Core: Drive'dan Veri Oku + RWKV Eğit

Drive'daki datasetleri oku, BPE tokenize et, RWKV modelini eğit.

**Önce `download_to_drive.py` ile verileri Drive'a indirin.**
Sonra bu notebook'u çalıştırın.

**Modeller:** rwkv_10m (36M), rwkv_50m (74M), rwkv_200m (252M), rwkv_1b (780M), rwkv_7b (7.25B)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. KURULUM
# ═══════════════════════════════════════════════════════════
!pip install torch numpy tqdm datasets tokenizers huggingface_hub

import torch, gc, json, time, os, math, random
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

# Drive mount
from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. AYARLAR
# ═══════════════════════════════════════════════════════════

# --- Drive yolu (download_to_drive.py'nin çıktığı yer) ---
DRIVE_DATA = Path('/content/drive/MyDrive/KRM-Core/Datasets')

# --- Model seçimi ---
MODEL = 'rwkv_50m'  # rwkv_10m, rwkv_50m, rwkv_200m, rwkv_1b, rwkv_7b

MODEL_CONFIGS = {
    'rwkv_10m':  {'d_model': 384,  'n_layers': 6,  'd_ff': 1536},
    'rwkv_50m':  {'d_model': 640,  'n_layers': 12, 'd_ff': 2560},
    'rwkv_200m': {'d_model': 1024, 'n_layers': 16, 'd_ff': 4096},
    'rwkv_1b':   {'d_model': 1536, 'n_layers': 24, 'd_ff': 6144},
    'rwkv_7b':   {'d_model': 4096, 'n_layers': 32, 'd_ff': 16384},
}

# --- Eğitim hyperparameters ---
BATCH_SIZE = 8
SEQ_LEN = 512
MAX_STEPS = 2000
LR = 3e-4
VOCAB_SIZE = 32768
SAVE_EVERY = 200

# --- Hangi datasetleri kullan ---
USE_DATASETS = ['turkce_wikipedia', 'turkce_oscar', 'math_gsm8k',
                'math_metamathqa', 'cs_the_stack_python', 'genel_fineweb_edu']

cfg = MODEL_CONFIGS[MODEL]
print(f"Model: {MODEL} (d={cfg['d_model']}, L={cfg['n_layers']}, ff={cfg['d_ff']})")
print(f"Veri dizini: {DRIVE_DATA}")
print(f"Kullanılacak datasetler: {USE_DATASETS}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. RWKV MODELİ
# ═══════════════════════════════════════════════════════════

class WKVMemory(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.log_gain = nn.Parameter(torch.zeros(d_model))
        self.log_decay = nn.Parameter(torch.zeros(d_model))

    def forward(self, values, keys):
        B, S, D = values.shape
        gain = torch.exp(self.log_gain)
        decay = torch.exp(-torch.exp(self.log_decay))
        out = torch.zeros_like(values)
        s = torch.zeros(B, D, device=values.device)
        n = torch.zeros(B, D, device=values.device)
        for t in range(S):
            k = keys[:, t] * gain
            s = s * decay + k * values[:, t]
            n = n * decay + k
            out[:, t] = s / (n + 1e-8)
        return out


class RWKVBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.time_k = nn.Linear(d_model, d_model, bias=False)
        self.time_v = nn.Linear(d_model, d_model, bias=False)
        self.time_r = nn.Linear(d_model, d_model, bias=False)
        self.time_o = nn.Linear(d_model, d_model, bias=False)
        self.wkv = WKVMemory(d_model)
        self.chan_k = nn.Linear(d_model, d_ff, bias=False)
        self.chan_v = nn.Linear(d_ff, d_model, bias=False)
        self.chan_r = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        r = x
        h = self.ln1(x)
        k, v, rg = self.time_k(h), self.time_v(h), self.time_r(h)
        h = r + torch.sigmoid(rg) * self.time_o(self.wkv(v, k))
        r = h
        h = self.ln2(h)
        k = F.relu(self.chan_k(h)) ** 2
        h = r + torch.sigmoid(self.chan_r(h)) * self.chan_v(k)
        return h


class RWKVLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_ff):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([RWKVBlock(d_model, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.embed(input_ids)
        for b in self.blocks:
            x = b(x)
        return self.head(self.norm(x))


print("RWKV modeli tanımlandı.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. DRIVE'DAN VERİ YÜKLE
# ═══════════════════════════════════════════════════════════

# BPE tokenizer eğit (veya varsa yükle)
from tokenizers import Tokenizer, trainers, pre_tokenizers, normalizers, decoders, processors
from tokenizers.models import BPE
from tokenizers import AddedToken

SPECIAL_TOKENS = [
    '<|SOURCE|>', '<|QUERY|>', '<|CONTEXT|>', '<|CONCEPTS|>', '<|EDGES|>',
    '<|HOT_GRAPH|>', '<|POLICY|>', '<|ANSWER_PLAN|>', '<|ANSWER|>', '<|INTENT|>',
    '<|DOMAIN|>', '<|SHARD|>', '<|UNCERTAIN|>', '<|SPECULATIVE|>', '<|DO_NOT_CLAIM|>',
    '<|SUPPORTED_BY_SOURCE|>', '<|MISSING_INFO|>', '<|END|>',
    '<|PAD|>', '<|UNK|>', '<|BOS|>', '<|EOS|>'
]

class BPETokenizer:
    def __init__(self, tok):
        self._tok = tok
        self.vocab_size = tok.get_vocab_size()
    def encode(self, text):
        return self._tok.encode(text).ids
    def decode(self, ids):
        return self._tok.decode(ids)

# Tokenizer'ı Drive'dan yükle veya eğ
tok_path = DRIVE_DATA.parent / 'tokenizer.json'
if tok_path.exists():
    print(f"Tokenizer yüklendi: {tok_path}")
    tok = Tokenizer.from_file(str(tok_path))
else:
    print("Tokenizer eğitiliyor (vocab=32K)...")
    # Drive'daki tüm .jsonl dosyalarını topla
    corpus_files = list(DRIVE_DATA.glob('*.jsonl'))
    if not corpus_files:
        print("HATA: Drive'da veri bulunamadı!")
        print(f"Beklenen dizin: {DRIVE_DATA}")
        print("Önce download_to_drive.py ile veri indirin.")
    else:
        print(f"{len(corpus_files)} dosya bulundu, tokenizer eğitimi başlıyor...")
        tok = Tokenizer(BPE(unk_token='<|UNK|>'))
        tok.normalizer = normalizers.NFKC()
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tok.decoder = decoders.ByteLevel()
        tok.post_processor = processors.ByteLevel(trim_offsets=False)
        special = [AddedToken(t, single_word=False, normalized=False) for t in SPECIAL_TOKENS]
        tok.add_special_tokens(special)
        trainer = trainers.BpeTrainer(
            vocab_size=VOCAB_SIZE,
            min_frequency=2,
            special_tokens=[str(t) for t in special],
            show_progress=True,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        )
        tok.train([str(f) for f in corpus_files], trainer)
        tok.save(str(tok_path))
        print(f"Tokenizer kaydedildi: {tok_path}")

tokenizer = BPETokenizer(tok)
print(f"Vocab: {tokenizer.vocab_size:,}")

# ── Tüm corpus'u oku ve tokenize et ──
all_ids = []
print(f"\nDrive'dan veriler okunuyor...")
for ds_file in sorted(DRIVE_DATA.glob('*.jsonl')):
    ds_name = ds_file.stem
    if USE_DATASETS and ds_name not in USE_DATASETS:
        continue
    print(f"  {ds_name}...", end=" ")
    count = 0
    with open(ds_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            text = record.get('text', '')
            if not text:
                continue
            ids = tokenizer.encode(text)
            all_ids.extend(ids)
            all_ids.append(tokenizer.encode('<|END|>')[0])
            count += 1
    print(f"{count:,} örnek, {len(all_ids):,} token toplam")

print(f"\nToplam token: {len(all_ids):,} (~{len(all_ids)/1e6:.1f}M)")
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. DATASET OLUSTUR
# ═══════════════════════════════════════════════════════════

class TokenDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.seq_len = seq_len
        n = (len(tokens) - 1) // seq_len
        self.data = torch.tensor(tokens[:n * seq_len + 1], dtype=torch.long)
        self.n_samples = n
        print(f"  {n:,} chunk, {len(self.data):,} token")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.data[start:start + self.seq_len]
        y = self.data[start + 1:start + self.seq_len + 1]
        return x, y

print("Dataset oluşturuluyor...")
dataset = TokenDataset(all_ids, SEQ_LEN)
del all_ids
gc.collect()

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Loader: {len(loader)} batch")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. EGITIM
# ═══════════════════════════════════════════════════════════

model = RWKVLM(
    vocab_size=tokenizer.vocab_size,
    d_model=cfg['d_model'],
    n_layers=cfg['n_layers'],
    d_ff=cfg['d_ff'],
).to(device)

params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL} | {params:,} param ({params*4/1024/1024:.0f} MB FP32)")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, MAX_STEPS)

history = {'step': [], 'loss': [], 'ppl': [], 'lr': []}
best_loss = float('inf')
step = 0

# Checkpoint'den devam
ckpt_dir = DRIVE_DATA.parent / f'checkpoints_{MODEL}'
ckpt_dir.mkdir(parents=True, exist_ok=True)
resume_path = ckpt_dir / 'latest.pt'
if resume_path.exists():
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    step = ckpt.get('step', 0)
    best_loss = ckpt.get('best_loss', float('inf'))
    print(f"\nDevam ediliyor: step {step}, best_loss {best_loss:.4f}")

save_dir = DRIVE_DATA.parent / f'trained_{MODEL}'
save_dir.mkdir(parents=True, exist_ok=True)

print(f"\n{'='*60}")
print(f"EGITIM: {MODEL} | {MAX_STEPS} adım | {device}")
print(f"{'='*60}")

t0 = time.time()
model.train()

while step < MAX_STEPS:
    for x_batch, y_batch in loader:
        if step >= MAX_STEPS:
            break
        step += 1

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(x_batch)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y_batch.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history['step'].append(step)
        history['loss'].append(loss.item())
        history['ppl'].append(math.exp(min(loss.item(), 20)))
        history['lr'].append(optimizer.param_groups[0]['lr'])

        if step % 10 == 0:
            elapsed = time.time() - t0
            recent_loss = history['loss'][-10:]
            avg_loss = sum(recent_loss) / len(recent_loss)
            avg_ppl = math.exp(min(avg_loss, 20))
            print(f"  Step {step:5d}/{MAX_STEPS} | Loss: {avg_loss:.4f} | PPL: {avg_ppl:.2f} | LR: {optimizer.param_groups[0]['lr']:.2e} | {elapsed:.0f}s")

        if step % SAVE_EVERY == 0:
            # Checkpoint kaydet
            torch.save({
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'config': cfg,
                'step': step,
                'best_loss': best_loss,
                'vocab_size': tokenizer.vocab_size,
            }, resume_path)

            if loss.item() < best_loss:
                best_loss = loss.item()
                torch.save(model.state_dict(), save_dir / f'{MODEL}_best.pt')

            # Drive'a kaydet
            torch.save({
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'config': cfg,
                'step': step,
                'best_loss': best_loss,
                'vocab_size': tokenizer.vocab_size,
                'history': history,
            }, save_dir / f'{MODEL}_step{step}.pt')
            print(f"    [*] Kaydedildi: step {step}")

print(f"\n{'='*60}")
print(f"TAMAMLANDI!")
print(f"Step: {step} | Süre: {time.time()-t0:.0f}s")
print(f"Best Loss: {best_loss:.4f} (PPL: {math.exp(min(best_loss, 20)):.2f})")
print(f"Model: {params:,} param")
print(f"Kayıtlar: {save_dir}/")

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. ÜRETIM TESTİ
# ═══════════════════════════════════════════════════════════

model.eval()

prompts = [
    "Bir dikdörtgenin alan formülü nedir?",
    "Python'da fibonacci serisi nasıl yazılır?",
    "Türkiye'nin başkenti neresidir?",
    "What is the time complexity of binary search?",
    "Write a function to check if a number is prime:",
]

for prompt in prompts:
    print(f"\n{'─'*50}")
    print(f"Soru: {prompt}")
    print(f"Cevap: ", end="")
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    with torch.no_grad():
        for _ in range(300):
            logits = model(input_ids[:, -SEQ_LEN:])
            probs = F.softmax(logits[0, -1] / 0.8, dim=0)
            next_id = torch.multinomial(probs, 1).item()
            input_ids = torch.cat([input_ids, torch.tensor([[next_id]], device=device)], dim=1)
            if next_id == tokenizer.encode('<|END|>')[0]:
                break
    output = tokenizer.decode(input_ids[0].tolist())
    # Prompt'u temizle
    gen = output[len(prompt):]
    print(gen[:500])

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. SON KAYIT
# ═══════════════════════════════════════════════════════════

# En son modeli Drive'a kaydet
final_path = save_dir / f'{MODEL}_final.pt'
torch.save({
    'model_state': model.state_dict(),
    'config': cfg,
    'history': history,
    'best_loss': best_loss,
    'params': params,
    'vocab_size': tokenizer.vocab_size,
}, final_path)

# Config kaydet
with open(save_dir / 'config.json', 'w') as f:
    json.dump({
        'model': MODEL,
        **cfg,
        'vocab_size': tokenizer.vocab_size,
        'best_loss': best_loss,
        'final_step': step,
    }, f, indent=2)

# History kaydet
with open(save_dir / 'history.json', 'w') as f:
    json.dump(history, f)

print(f"\nDrive'a kaydedildi: {save_dir}/")
for f in sorted(save_dir.glob('*')):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f}MB)")

## Sonraki Adımlar

1. **Büyük model:** `rwkv_200m` veya `rwkv_1b` ile devam et
2. **Daha fazla veri:** `download_to_drive.py` ile tüm datasetleri indir
3. **Kavramsal Parçalı Eğitim:** Büyük modeli bloklara böl, her bloğu ayrı eğit
4. **Rezonans Motoru:** Eğitilen modeli KRM-Core inference sistemine entegre et